# Week 24, Day 4: Test-driven development

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def initials(name):
    raise NotImplementedError

def test_two_names():
    assert initials("Ada Lovelace") == "A.L."

try:
    test_two_names()
    print("passed")
except NotImplementedError:
    print("red: the test fails because the code doesn't exist yet")

In [ ]:
def price_label(pence):
    if pence < 100:
        return f"{pence}p"
    pounds, rest = divmod(pence, 100)
    label = f"£{pounds:,}"
    if rest:
        label += f".{rest:02d}"
    return label

def test_whole_pounds():
    assert price_label(100) == "£1"
    assert price_label(300) == "£3"

def test_pounds_and_pence():
    assert price_label(250) == "£2.50"
    assert price_label(205) == "£2.05"

def test_under_a_pound():
    assert price_label(5) == "5p"
    assert price_label(99) == "99p"

def test_thousands():
    assert price_label(123456) == "£1,234.56"

for t in (test_whole_pounds, test_pounds_and_pence, test_under_a_pound, test_thousands):
    t()
print("all green")

> **Key idea.** TDD repeats red (write a small failing test), green (the simplest code that passes) and refactor (tidy while the tests stay green). Each piece of code exists because a test needed it. Refactoring changes structure, never behaviour, and is only done on green, in small steps. TDD suits code whose results you can state in advance.

### Exercise: Make them green

The tests in `test_initials.py` were written first. Write `initials(name)` so they all pass. Your code is the module `names.py`, which the tests import.

In [ ]:
# Run this cell first: it creates the files used below.
with open('test_initials.py', 'w', encoding='utf-8') as f:
    f.write('from names import initials\n\ndef test_two_names():\n    assert initials("Ada Lovelace") == "A.L."\n\ndef test_one_name():\n    assert initials("Plato") == "P."\n\ndef test_lower_case():\n    assert initials("grace brewster hopper") == "G.B.H."\n\ndef test_extra_spaces():\n    assert initials("  Alan   Turing ") == "A.T."\n\ndef test_empty():\n    assert initials("") == ""\n')

In [ ]:
def initials(name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TESTS = open(\"test_initials.py\").read()\ndef test_green():\n    \"all the tests in test_initials.py pass\"\n    r = pytest_run(TESTS, files={\"names.py\": source})\n    assert not r[\"failed\"] and len(r[\"passed\"]) == 5, f\"failing: {r['failed']}\"")

### Exercise: Red first

Before anyone writes `is_valid_postcode_area(text)` in `postcodes.py`, write its tests. The rule: one or two letters, then one or two digits (for example `"M1"`, `"SW19"`, `"B33"`, `"EH1"`), capital letters only, nothing else. The function returns True or False. Your tests must **fail** on today's stub (that's the red step) and **pass** on a correct version written later, and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('postcodes.py', 'w', encoding='utf-8') as f:
    f.write('def is_valid_postcode_area(text):\n    raise NotImplementedError\n')

In [ ]:
from postcodes import is_valid_postcode_area

In [ ]:
# Check: run the exercise cell above first, then this one.
check("STUB = open(\"postcodes.py\").read()\nGOOD = \"import re\\ndef is_valid_postcode_area(text):\\n    return re.fullmatch(r'[A-Z]{1,2}[0-9]{1,2}', text) is not None\\n\"\nBROKEN = {\n    \"allows three letters\": GOOD.replace(\"{1,2}[0-9]\", \"{1,3}[0-9]\"),\n    \"allows lower case\": GOOD.replace(\"[A-Z]\", \"[A-Za-z]\"),\n    \"no digits needed\": GOOD.replace(\"[0-9]{1,2}\", \"[0-9]{0,2}\"),\n    \"allows three digits\": GOOD.replace(\"[0-9]{1,2}\", \"[0-9]{1,3}\"),\n    \"search not fullmatch\": GOOD.replace(\"fullmatch\", \"match\"),\n}\ndef test_red():\n    \"every one of your tests fails on the stub\"\n    r = against(source, \"postcodes\", {\"stub\": STUB})\n    passed = pytest_run(source)[\"passed\"]\n    write_file(\"postcodes.py\", STUB)\n    assert r[\"stub\"] and not passed, \"a test that passes before the code exists isn't testing it\"\ndef test_green_later():\n    \"your tests pass on a correct version and catch every broken one\"\n    results = against(source, \"postcodes\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"postcodes.py\", STUB)\n    assert results[\"correct\"] == [], f\"these fail on a correct version: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Grow it test by test

The tests in `test_roman.py` were written in TDD order, simplest first. Write `to_roman(n)` (in `roman.py`) to pass them all. Try working through them in order, getting each one green before the next.

In [ ]:
# Run this cell first: it creates the files used below.
with open('test_roman.py', 'w', encoding='utf-8') as f:
    f.write('import pytest\nfrom roman import to_roman\n\ndef test_one():\n    assert to_roman(1) == "I"\n\ndef test_three():\n    assert to_roman(3) == "III"\n\ndef test_five_and_ten():\n    assert to_roman(5) == "V"\n    assert to_roman(10) == "X"\n\ndef test_adding():\n    assert to_roman(8) == "VIII"\n    assert to_roman(27) == "XXVII"\n\ndef test_subtractive():\n    assert to_roman(4) == "IV"\n    assert to_roman(9) == "IX"\n    assert to_roman(40) == "XL"\n    assert to_roman(90) == "XC"\n\ndef test_large():\n    assert to_roman(1994) == "MCMXCIV"\n    assert to_roman(2026) == "MMXXVI"\n    assert to_roman(3999) == "MMMCMXCIX"\n\n@pytest.mark.parametrize("n", [0, -1, 4000])\ndef test_out_of_range(n):\n    with pytest.raises(ValueError):\n        to_roman(n)\n')

In [ ]:
def to_roman(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TESTS = open(\"test_roman.py\").read()\ndef test_green():\n    \"all the tests in test_roman.py pass\"\n    r = pytest_run(TESTS, files={\"roman.py\": source})\n    assert not r[\"failed\"] and len(r[\"passed\"]) == 9, f\"failing: {r['failed']}\"")

### Exercise: Refactor on green

This works, and `test_shipping.py` passes. Refactor `shipping_days` (in `shipping.py`) to replace the `if`/`elif` chain with a dictionary lookup, without changing its behaviour: the tests must stay green, and there must be no `elif` left.

In [ ]:
# Run this cell first: it creates the files used below.
with open('test_shipping.py', 'w', encoding='utf-8') as f:
    f.write('import pytest\nfrom shipping import shipping_days\n\n@pytest.mark.parametrize("country, days", [("UK", 1), ("uk", 1), ("FR", 3), ("DE", 3), ("US", 7), ("NG", 10), ("JP", 10)])\ndef test_days(country, days):\n    assert shipping_days(country) == days\n')

In [ ]:
def shipping_days(country):
    country = country.upper()
    if country == "UK":
        return 1
    elif country == "FR":
        return 3
    elif country == "DE":
        return 3
    elif country == "US":
        return 7
    else:
        return 10

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TESTS = open(\"test_shipping.py\").read()\ndef test_green():\n    \"test_shipping.py still passes\"\n    r = pytest_run(TESTS, files={\"shipping.py\": source})\n    assert not r[\"failed\"] and len(r[\"passed\"]) == 7, f\"failing: {r['failed']}\"\ndef test_refactored():\n    \"uses a dictionary, with no elif\"\n    assert \"elif\" not in source and \"{\" in source")

### Exercise: Name the step



In [ ]:
a = ___   # the three steps of the TDD cycle, in order, as a list of lower-case words
b = ___   # the step in which you're allowed to hard-code a return value
c = ___   # the step in which behaviour must not change
d = ___   # True or False: in TDD, a new test that passes straight away is a good sign

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [\"red\", \"green\", \"refactor\"] and b == \"green\" and c == \"refactor\" and d is False")

### Bug hunt: Faked too long

`average` passes its one test because it was faked in the green step, and nobody added the tests that would force the real code. Write the real implementation (the mean of a list of numbers, with an empty list raising `ValueError`), and add tests that would have caught the fake.

In [ ]:
def average(numbers):
    return 2

def test_average():
    assert average([1, 2, 3]) == 2

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_real():\n    \"average works for any list\"\n    assert average([1, 2, 3]) == 2 and average([10]) == 10 and average([1, 2]) == 1.5 and average([-4, 4, 6]) == 2\n    try:\n        average([])\n        assert False, \"average([]) should raise ValueError\"\n    except ValueError:\n        pass\ndef test_catch_fake():\n    \"your tests catch the fake\"\n    import ast\n    tests_only = \"import pytest\\nfrom avg import average\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    r = against(tests_only, \"avg\", {\"fake\": \"def average(numbers):\\n    return 2\\n\",\n                                    \"real\": \"def average(numbers):\\n    if not numbers:\\n        raise ValueError()\\n    return sum(numbers) / len(numbers)\\n\"})\n    assert r[\"real\"] == [] and r[\"fake\"], \"your tests should pass on a real average and fail on the fake\"")

### Bug hunt: Refactored and broken

Someone "tidied" `vowel_count` and broke it, but didn't run the tests. Run them (press Check), find which case fails, and fix the function. Keep the tidier style.

In [ ]:
def vowel_count(text):
    return sum(1 for c in text if c in "aeiou")

def test_vowels():
    assert vowel_count("banana") == 3
    assert vowel_count("rhythm") == 0
    assert vowel_count("AEIOU and you") == 8

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_green():\n    \"the tests pass\"\n    r = pytest_run(source)\n    assert r[\"passed\"] == [\"test_vowels\"], \"test_vowels still fails\"\ndef test_still_tidy():\n    \"still one line with sum\"\n    assert \"sum(\" in source")

### Puzzle: One TDD round

Arrange the lines: the test (written first), then the simplest code that passes it. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: TDD a word wrapper

Write `wrap(text, width)` and its tests, test first. It splits text into lines of at most `width` characters, breaking only between words, and returns a list of lines. Words are separated by any whitespace; lines don't start or end with a space. A word longer than `width` goes on a line by itself (not split). Empty text gives `[]`. A width under 1 raises `ValueError`. For example `wrap("the quick brown fox", 10)` is `["the quick", "brown fox"]`.

Your one editor holds both your function and your tests. The checks run your tests against your function and against six broken versions, and run their own hidden tests on your function.

In [ ]:
import pytest

def test_example():
    assert wrap("the quick brown fox", 10) == ["the quick", "brown fox"]

def wrap(text, width):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast as _ast\nGOOD = '''\ndef wrap(text, width):\n    if width < 1:\n        raise ValueError(\"width must be at least 1\")\n    lines, line = [], \"\"\n    for word in text.split():\n        if not line:\n            line = word\n        elif len(line) + 1 + len(word) <= width:\n            line += \" \" + word\n        else:\n            lines.append(line)\n            line = word\n    if line:\n        lines.append(line)\n    return lines\n'''\nBROKEN = {\n    \"off by one\": GOOD.replace(\"<= width\", \"< width\"),\n    \"loses last line\": GOOD.replace(\"    if line:\\n        lines.append(line)\\n\", \"\"),\n    \"splits on single spaces\": GOOD.replace(\"text.split()\", 'text.split(\" \")'),\n    \"empty gives ['']\": GOOD.replace(\"    if line:\\n        lines.append(line)\\n\", \"    lines.append(line)\\n\"),\n    \"width 0 allowed\": GOOD.replace(\"width < 1\", \"width < 0\"),\n    \"long word breaks badly\": GOOD.replace(\"        if not line:\\n            line = word\\n        elif\", \"        if not line and len(word) <= width:\\n            line = word\\n        elif line and\"),\n}\ndef test_hidden():\n    \"your wrap passes the hidden tests\"\n    assert wrap(\"the quick brown fox\", 10) == [\"the quick\", \"brown fox\"]\n    assert wrap(\"\", 5) == [] and wrap(\"   \", 5) == []\n    assert wrap(\"a b c\", 3) == [\"a b\", \"c\"]\n    assert wrap(\"one two\", 7) == [\"one two\"]\n    assert wrap(\"extraordinary day\", 5) == [\"extraordinary\", \"day\"]\n    assert wrap(\"  spaced\\tout\\nwords \", 9) == [\"spaced\", \"out words\"]\n    for w in (0, -2):\n        try:\n            wrap(\"x\", w)\n            assert False, f\"wrap('x', {w}) should raise ValueError\"\n        except ValueError:\n            pass\ndef test_yours():\n    \"your tests catch all six broken versions\"\n    tree = _ast.parse(source)\n    tests_only = \"import pytest\\nfrom wrapper import wrap\\n\" + \"\\n\".join(\n        _ast.unparse(n) for n in tree.body if isinstance(n, _ast.FunctionDef) and n.name.startswith(\"test_\"))\n    results = against(tests_only, \"wrapper\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on a correct wrap: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

## Warm-up

In TDD, this is where you start: a test for code that isn't written yet, failing on purpose. A failing test you've just watched fail is a test you know can fail, which is more than many tests can say.

## Red, green, refactor

TDD repeats a short cycle, usually a few minutes long:

1. **Red.** Write one small test for the next bit of behaviour, and run it. It must **fail**. If it passes already, either the behaviour exists or the test doesn't check what you think.
2. **Green.** Write the **simplest** code that makes it pass. Simplest really means simplest, even if it looks naive; the next tests will push it towards the general solution.
3. **Refactor.** With all tests passing, tidy the code (and the tests): better names, less repetition, clearer structure. Run the tests after each change. They must stay green.

Then pick the next behaviour and go round again.

Why bother?

- You decide **what** the code should do before deciding **how**, which catches unclear requirements early.
- You work in small steps. When something breaks, it was the last few lines you wrote.
- Every behaviour has a test, because no code was written without one.
- Code written to be tested tends to be simpler to use: small functions, clear inputs and outputs.

## A worked example

The task: `price_label(pence)` turns a price in pence into a label. 250 becomes `"£2.50"`, 5 becomes `"5p"`, 100 becomes `"£1"` (no `.00`), and 123456 becomes `"£1,234.56"`.

**Round 1.** Red: the simplest case first.

```
def test_whole_pounds():
    assert price_label(100) == "£1"
```

Green: the simplest code that passes. Hard-coding it is fine; it's honest about what the tests demand so far.

```
def price_label(pence):
    return "£1"
```

**Round 2.** Red: a second whole-pound case forces real code.

```
def test_more_whole_pounds():
    assert price_label(300) == "£3"
```

Green:

```
def price_label(pence):
    return f"£{pence // 100}"
```

**Round 3.** Red: pounds and pence.

```
def test_pounds_and_pence():
    assert price_label(250) == "£2.50"
```

Green:

```
def price_label(pence):
    pounds, rest = divmod(pence, 100)
    if rest == 0:
        return f"£{pounds}"
    return f"£{pounds}.{rest:02d}"
```

**Round 4.** Red: under a pound. Green: one more branch. **Round 5.** Red: thousands. Green: the `,` format option. Then refactor. Here's where it ends up, with all its tests:

Notice `price_label(205)`: the `:02d` format is there because a test demanded `"£2.05"` rather than `"£2.5"`. Each part of the code exists because a test asked for it.

## Refactoring under green

Refactoring means changing the **structure** of code without changing its **behaviour**. Tests are what make it safe: if they stay green, the behaviour hasn't changed.

Good refactoring habits:

- **Only refactor on green.** Never tidy and add features in the same step; if the tests then fail, you won't know which change caused it.
- **Small steps, test after each**: rename a variable, run; extract a function, run.
- **Refactor tests too**: repeated setup can move into a helper (or a fixture, tomorrow), and many similar tests can become one parametrized test.

Common refactorings: renaming, extracting a function, replacing a chain of `if`/`elif` with a dictionary lookup, removing duplication, replacing a loop with a comprehension or a built-in like `sum` or `any`.

## When TDD helps

TDD works best when you know **what** the result should be, but not yet **how** to produce it: parsing, calculations, rules, data transformations, fixing a bug (write the failing regression test first, as on Day 1).

It helps less when:

- you're exploring and don't yet know what you want: try things in a notebook or the playground first, then write tests once the design settles;
- the result is something you judge by eye, such as a chart's look or a page's layout;
- the code is mostly glue to things outside your control (a website, a database); test the logic around it instead.

Even where you don't follow it strictly, the habits carry over: think of the test first, take small steps, and keep the tests green while you tidy.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a small function you'd like to have (a slug maker that turns "Hello, World!" into `"hello-world"`, say) and build it test first: one failing test, the simplest code, tidy, repeat. Notice how often the next test changes your mind about the design. Tomorrow: edge cases, and pytest fixtures for sharing setup between tests.